# Silver Layer: Full Load
**Source:** `bronze` | **Target:** `silver`

**Method:** CTAS (Create Table As Select)

## Table: silver_crm_customer_info

In [ ]:
%%sql

CREATE OR REPLACE TABLE sales_lakehouse.dbo.silver_crm_customer_info
AS
    SELECT
        CAST(cst_id AS INT) AS cst_id,
        CAST(cst_key AS STRING) AS cst_key,
        -- removing space from firstname and lastname
        TRIM(cst_firstname) AS cst_firstname,
        TRIM(cst_lastname) AS cst_lastname,
        -- data standardization & consistency
        CASE UPPER(TRIM(cst_marital_status))
            WHEN 'M' THEN 'Married'
            WHEN 'S' THEN 'Single'
            ELSE 'Unknown'
        END AS cst_marital_status,
        CASE UPPER(TRIM(cst_gndr))
            WHEN 'M' THEN 'Male'
            WHEN 'F' THEN 'Female'
            ELSE 'Unknown'
        END AS cst_gndr,
        -- casting to date
        CAST(cst_create_date AS DATE) AS cst_create_date,
        -- adding metadata columns
        CURRENT_TIMESTAMP() AS meta_load_timestamp,
        'CRM' AS meta_source_system,
        'crm_customer_info' AS meta_table_name
    FROM
    -- subquery to filter only the latest cst_id
    (
        SELECT 
        *,
        ROW_NUMBER() OVER(PARTITION BY cst_id ORDER BY cst_create_date DESC) AS flag_last
        FROM sales_lakehouse.dbo.bronze_crm_customer_info
        WHERE cst_id IS NOT NULL
    )t
    WHERE flag_last = 1; -- considering only the latest cst_ids

## Table: silver_crm_product_info

In [ ]:
CREATE OR REPLACE TABLE sales_lakehouse.dbo.silver_crm_product_info
AS
SELECT 
    prd_id,
    
    REPLACE(SUBSTRING(prd_key, 1, 5), "-", "_") AS cat_id,      -- key to join with erp_product_category
    
    SUBSTRING(prd_key, 7, LEN(prd_key)) AS prd_key,             -- key to join with crm_sales_details
    
    prd_nm,
    
    CAST(COALESCE(prd_cost, 0) AS INT) AS prd_cost,             -- replacing NULL with 0
    
    -- Data Standardization
    CASE UPPER(TRIM(prd_line))
        WHEN 'M' THEN 'Mountain'
        WHEN 'S' THEN 'Other Sales'
        WHEN 'R' THEN 'Road'
        WHEN 'T' THEN 'Touring'
        ELSE 'Unknown'
    END AS prd_line,

-- End Date = Start Date of the Next Record - 1
    CAST(prd_start_dt AS DATE) AS prd_start_dt,
    CAST(LEAD(prd_start_dt) OVER(PARTITION BY prd_key ORDER BY prd_start_dt) AS DATE) - 1 AS prd_end_dt, -- adding -1 to avoid date overlapping

-- adding metadata columns
    CURRENT_TIMESTAMP() AS meta_load_timestamp,
    'CRM' AS meta_source_system,
    'crm_product_info' AS meta_table_name
FROM sales_lakehouse.dbo.bronze_crm_product_info;

## Table: silver_crm_sales_details

In [ ]:
-- Creating silver_crm_sales_details table
CREATE OR REPLACE TABLE sales_lakehouse.dbo.silver_crm_sales_details
AS
SELECT 
    sls_ord_num,
    sls_prd_key,
    CAST(sls_cust_id AS INT) AS sls_cust_id,

    -- Data handling
    CASE WHEN sls_order_dt = 0 OR LEN(sls_order_dt) !=8 THEN NULL
         ELSE TO_DATE(CAST(sls_order_dt AS STRING), 'yyyyMMdd')
    END AS sls_order_dt,

-- applying same conditions for future proof
    CASE WHEN sls_ship_dt = 0 OR LEN(sls_ship_dt) !=8 THEN NULL
         ELSE TO_DATE(CAST(sls_ship_dt AS STRING), 'yyyyMMdd')
    END AS sls_ship_dt,

-- applying same conditions for future proof
    CASE WHEN sls_due_dt = 0 OR LEN(sls_due_dt) !=8 THEN NULL
         ELSE TO_DATE(TRIM(CAST(sls_due_dt AS STRING)), 'yyyyMMdd')
    END AS sls_due_dt,

-- applying business rules for sales, quantity and price
    CAST(
        CASE WHEN sls_sales <= 0 OR sls_sales IS NULL OR sls_sales != sls_quantity * ABS(sls_price)
            THEN sls_quantity * ABS(sls_price)
            ELSE sls_sales
        END 
    AS INT) AS sls_sales,

    CAST(sls_quantity AS INT) AS sls_quantity,

    CAST(
        CASE WHEN sls_price <= 0 OR sls_price IS NULL
            THEN sls_sales / NULLIF(sls_quantity,0)       -- handling divide by 0 error
            ELSE sls_price
        END
    AS INT) AS sls_price,

-- adding metadata columns
    CURRENT_TIMESTAMP() AS meta_load_timestamp,
    'CRM' AS meta_source_system,
    'crm_sales_details' AS meta_table_name

FROM sales_lakehouse.dbo.bronze_crm_sales_details;

## Table: silver_erp_customers

In [ ]:
CREATE OR REPLACE TABLE sales_lakehouse.dbo.silver_erp_customers
AS
SELECT
-- removing 'NAS' from cid if it is there 
    CASE WHEN cid LIKE 'NAS%' THEN SUBSTRING(cid, 4, LEN(cid))
         ELSE cid
    END AS cid,

-- replacing future birth dates with NULL
    CAST(
        CASE WHEN bdate > CURRENT_TIMESTAMP() THEN NULL
            ELSE bdate
        END 
    AS DATE) AS bdate,

-- Data Standaridization 
    CASE WHEN UPPER(TRIM(gen)) IN ('M', 'MALE') THEN 'Male'
         WHEN UPPER(TRIM(gen)) IN ('F', 'FEMALE') THEN 'Female'
         ELSE 'Unknown'
    END AS gen,

-- adding metadata columns
    CURRENT_TIMESTAMP() AS meta_load_timestamp,
    'ERP' AS meta_source_system,
    'erp_customers' AS meta_table_name

FROM sales_lakehouse.dbo.bronze_erp_customers;

## Table: silver_erp_location

In [ ]:
CREATE OR REPLACE TABLE sales_lakehouse.dbo.silver_erp_location
SELECT
-- fixing cid column (key column)
    REPLACE(cid, '-', '') AS cid,

-- Data Standardization
    CASE WHEN TRIM(cntry) = 'DE' THEN 'Germany'
        WHEN TRIM(cntry) IN ('US', 'USA') THEN 'United States'
        WHEN TRIM(cntry) = '' OR cntry IS NULL THEN 'Unknown'
        ELSE cntry
    END AS cntry,

    -- adding metadata columns
        CURRENT_TIMESTAMP() AS meta_load_timestamp,
        'ERP' AS meta_source_system,
        'erp_location' AS meta_table_name

FROM sales_lakehouse.dbo.bronze_erp_location;

## Table: silver_erp_product_category

In [ ]:
CREATE OR REPLACE TABLE sales_lakehouse.dbo.silver_erp_product_category
AS
SELECT
    id,
    cat,
    subcat,
    maintenance
FROM sales_lakehouse.dbo.bronze_erp_product_category;

### Updating the watermark / Incremental Load Control table with the maximum order_dt from sales_details

In [ ]:
UPDATE sales_lakehouse.dbo.incremental_load_control
SET last_loaded_value = COALESCE(
    (SELECT MAX(sls_order_dt) 
    FROM sales_lakehouse.dbo.silver_crm_sales_details),
    last_loaded_value
)
WHERE table_name = 'crm_sales_details';